# Build sourmash trees of Cryptococcus species complex genomes, and compare to BUSCO trees visually and using Robinson-Foulds distance

## ____________________________________________________________________________________________

### Import dependencies, and load the tree build using BUSCOs

In [ ]:
## Import dependencies

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from Bio import Phylo
from Bio.Phylo.TreeConstruction import DistanceMatrix, DistanceTreeConstructor
from Bio.Phylo import draw
import pickle
from dendropy.calculate import treecompare
from dendropy import Tree, TaxonNamespace
import networkx as nx
from collections import defaultdict
import matplotlib.colors as mcolors
from pathlib import Path
import scipy.stats
import math
import random

# Directory for saving sourmash trees
SourmashTreeDir = Path("/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashTrees")

In [ ]:
## Load the BUSCO tree

# Switch to directory containing the tree
os.chdir('/path/to/your/analysis_directory/ForGeneTrees/IntermediateFiles/Cryptococcus_IF')

# Load dictionary from a file
with open('CryptococcusDictionary.pkl', 'rb') as file:
    CryptococcusDictionary = pickle.load(file)


# Load the IQ-TREE output file
os.chdir('/path/to/your/analysis_directory/ForGeneTrees/IntermediateFiles/Cryptococcus_Reduced_IF')
BUSCOtree = Phylo.read("Cryptococcus_Reduced_BUSCOsTrimmed_ForOutgroup.faa.treefile", "newick")

# Update the leaf labels in the tree
for leaf in BUSCOtree.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]

# Ladderize tree (re-organize leaves)
BUSCOtree.ladderize(reverse=True)

# Plot the updated tree
fig = plt.figure(figsize=(6, 9))
ax = fig.add_subplot(1, 1, 1)

# Draw the tree without showing it yet
Phylo.draw(BUSCOtree, axes=ax, do_show=False)

# Adjust font sizes for all labels
for text in ax.texts:
    text.set_fontsize(8) 

# Add a title
fig.suptitle("Maximum Likelihood BUSCO Tree", fontsize=12, fontweight='bold')

plt.show()

In [ ]:
os.chdir('/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashTrees')


# Write the corrected tree to a new Newick file for Robinson-Foulds computation
Phylo.write(BUSCOtree, "CryptococcusBUSCOs_corrected_names.tree", "newick")

In [ ]:
# Function to hide internal node labels
def hide_inner_labels(clade):
    return "" if not clade.is_terminal() else clade.name

In [ ]:
from scipy.stats import spearmanr

def tree_distance_correlation(tree1, tree2, method='spearman'):
    """
    Compute correlation between pairwise distances of two DendroPy trees.
    
    Args:
        tree1: DendroPy Tree object
        tree2: DendroPy Tree object
        method (str): 'pearson' or 'spearman' (default)
    
    Returns:
        tuple: (correlation_coefficient, p_value, log10_p_value, number_of_pairs)
    """
    # Verify taxa match
    taxa1 = sorted([taxon.label for taxon in tree1.taxon_namespace])
    taxa2 = sorted([taxon.label for taxon in tree2.taxon_namespace])
    if taxa1 != taxa2:
        raise ValueError("Trees must have identical taxa!")
    
    # Get all taxa as a list
    taxa = list(tree1.taxon_namespace)
    n = len(taxa)
    i, j = np.tril_indices(n, k=-1)  # Lower triangle indices
    
    # Compute pairwise distances
    pdm1 = tree1.phylogenetic_distance_matrix()
    pdm2 = tree2.phylogenetic_distance_matrix()
    
    # Extract distances using proper taxon access
    d1 = np.array([pdm1.distance(taxa[x], taxa[y]) for x, y in zip(i, j)])
    d2 = np.array([pdm2.distance(taxa[x], taxa[y]) for x, y in zip(i, j)])
    n_pairs = len(d1)
    
    # Compute correlation
    if method == 'pearson':
        from scipy.stats import pearsonr
        corr, pval = pearsonr(d1, d2)
    elif method == 'spearman':
        corr, pval = spearmanr(d1, d2)
    else:
        raise ValueError("Method must be 'pearson' or 'spearman'")
    
    log_pval = np.log10(pval) if pval > 0 else -np.inf
    
    return corr, pval, log_pval, n_pairs

## ____________________________________________________________________________________________

### Load the sourmash matrices and convert to trees, also prepare a combine k21 and k51 matrix (where under 95% ANI k=21, and over 95% k=51)
### Starting with Jaccard similarity (the later, by max containment)

##### First load matrices for sourmash ANI computed by Jaccard

In [ ]:
# Load the k=21 and k=51 ANI matrices
k16_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k16/comprehensive_jaccard.npy"
k21_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k21/comprehensive_jaccard.npy"
k31_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k31/comprehensive_jaccard.npy"
k51_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k51/comprehensive_jaccard.npy"
k61_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k61/comprehensive_jaccard.npy"
k71_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k71/comprehensive_jaccard.npy"

ani_16_j = np.load(k16_path_j)
ani_21_j = np.load(k21_path_j)
ani_31_j = np.load(k31_path_j)
ani_51_j = np.load(k51_path_j)
ani_61_j = np.load(k61_path_j)
ani_71_j = np.load(k71_path_j)

# Load the labels for both k values
k16_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k16/comprehensive_jaccard.npy.labels.txt"
k21_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k21/comprehensive_jaccard.npy.labels.txt"
k31_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k31/comprehensive_jaccard.npy.labels.txt"
k51_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k51/comprehensive_jaccard.npy.labels.txt"
k61_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k61/comprehensive_jaccard.npy.labels.txt"
k71_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k71/comprehensive_jaccard.npy.labels.txt"

def parse_labels(label_file):
    with open(label_file) as f:
        labels = [line.strip() for line in f]
    # Parse labels to get just the accession (GCA_XXXXXX.X)
    parsed_labels = []
    for label in labels:
        # Get the filename part after the last '/'
        filename = label.split('/')[-1]
        # Split by '_' and take first two parts
        accession = '_'.join(filename.split('_')[:2])
        parsed_labels.append(accession)
    return parsed_labels

k16_labels_j = parse_labels(k16_labels_path_j)
k21_labels_j = parse_labels(k21_labels_path_j)
k31_labels_j = parse_labels(k31_labels_path_j)
k51_labels_j = parse_labels(k51_labels_path_j)
k61_labels_j = parse_labels(k61_labels_path_j)
k71_labels_j = parse_labels(k71_labels_path_j)


In [ ]:
## Compute the merged ANI matrix for k=21 until ANI becomes 95%, then switch to k=51

# Verify the labels match between k21 and k51
if k21_labels_j != k51_labels_j:
    print("Warning: Label order differs between k21 and k51 matrices!")
    
    # Create a mapping from accession to index in k51 matrix
    k51_label_to_index_j = {label: i for i, label in enumerate(k51_labels_j)}
    
    # Reorder the k51 matrix to match k21's order
    reorder_indices_j = [k51_label_to_index_j[label] for label in k21_labels_j]
    ani_51_j_reorder = ani_51_j[reorder_indices_j][:, reorder_indices_j]
    
    print("Reordered k51 matrix to match k21 label order")
else:
    print("Label orders match between k21 and k51 matrices")

# Merge the matrices
k21k51_ani_j = np.where(ani_21_j > 0.95, ani_51_j_reorder, ani_21_j)

# Save the merged matrix as both .npy and .csv
output_dir = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k21k51"
os.makedirs(output_dir, exist_ok=True)
np.save(f"{output_dir}/k21and51_ANI_jaccard.npy", k21k51_ani_j)

# Save as CSV with parsed labels
df = pd.DataFrame(k21k51_ani_j, index=k21_labels_j, columns=k21_labels_j)
df.to_csv(f"{output_dir}/k21and51_ANI_jaccard_parsed.csv")

##### Now build each sourmash tree

In [ ]:
## For k=16

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_16_j

# Condense to lower-triangular format
num_taxa = len(ani_16_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(k16_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_16 = constructor.upgma(dist_matrix)
upgma_tree_16.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_16.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k16_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_16, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=21

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_21_j

# Condense to lower-triangular format
num_taxa = len(ani_21_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k21_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_21 = constructor.upgma(dist_matrix)
upgma_tree_21.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_21.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]



## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k21_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_21, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=31

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_31_j

# Condense to lower-triangular format
num_taxa = len(ani_31_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k31_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_31 = constructor.upgma(dist_matrix)
upgma_tree_31.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_31.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]



## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k31_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_31, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=51

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_51_j

# Condense to lower-triangular format
num_taxa = len(ani_51_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k51_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_51 = constructor.upgma(dist_matrix)
upgma_tree_51.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_51.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k51_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_51, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=61

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_61_j

# Condense to lower-triangular format
num_taxa = len(ani_61_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k61_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_61 = constructor.upgma(dist_matrix)
upgma_tree_61.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_61.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k61_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_61, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=71

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_71_j

# Condense to lower-triangular format
num_taxa = len(ani_71_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k71_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_71 = constructor.upgma(dist_matrix)
upgma_tree_71.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_71.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k71_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_71, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=21 until ANI becomes 95%, then switch to k=51

# Convert similarity matrix to distance matrix
DistMatrix = 1 - k21k51_ani_j

# Condense to lower-triangular format
num_taxa = len(k21k51_ani_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k21_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_21and51 = constructor.upgma(dist_matrix)
upgma_tree_21and51.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_21and51.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]



## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k21andk51_jaccard.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_21and51, output_path, "newick")

print(f"Tree saved to: {output_path}")

## ____________________________________________________________________________________________

### Plot sourmash trees next to BUSCO trees, and compute Robinson-Foulds distances

In [ ]:
from scipy.stats import spearmanr

def tree_distance_correlation(tree1, tree2, method='spearman'):
    """
    Compute correlation between pairwise distances of two DendroPy trees.
    
    Args:
        tree1: DendroPy Tree object
        tree2: DendroPy Tree object
        method (str): 'pearson' or 'spearman' (default)
    
    Returns:
        tuple: (correlation_coefficient, p_value, log10_p_value, number_of_pairs)
    """
    # Verify taxa match
    taxa1 = sorted([taxon.label for taxon in tree1.taxon_namespace])
    taxa2 = sorted([taxon.label for taxon in tree2.taxon_namespace])
    if taxa1 != taxa2:
        raise ValueError("Trees must have identical taxa!")
    
    # Get all taxa as a list
    taxa = list(tree1.taxon_namespace)
    n = len(taxa)
    i, j = np.tril_indices(n, k=-1)  # Lower triangle indices
    
    # Compute pairwise distances
    pdm1 = tree1.phylogenetic_distance_matrix()
    pdm2 = tree2.phylogenetic_distance_matrix()
    
    # Extract distances using proper taxon access
    d1 = np.array([pdm1.distance(taxa[x], taxa[y]) for x, y in zip(i, j)])
    d2 = np.array([pdm2.distance(taxa[x], taxa[y]) for x, y in zip(i, j)])
    n_pairs = len(d1)
    
    # Compute correlation
    if method == 'pearson':
        from scipy.stats import pearsonr
        corr, pval = pearsonr(d1, d2)
    elif method == 'spearman':
        corr, pval = spearmanr(d1, d2)
    else:
        raise ValueError("Method must be 'pearson' or 'spearman'")
    
    log_pval = np.log10(pval) if pval > 0 else -np.inf
    
    return corr, pval, log_pval, n_pairs

In [ ]:
os.chdir(SourmashTreeDir)

# Create a shared TaxonNamespace
shared_taxon_namespace = TaxonNamespace()

# Load the trees with the shared TaxonNamespace
os.chdir(SourmashTreeDir)
newick_busco = Tree.get(path="CryptococcusBUSCOs_corrected_names.tree", schema="newick", taxon_namespace=shared_taxon_namespace)

        
newick_sourmash_k16 = Tree.get(path="upgma_tree_k16_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21 = Tree.get(path="upgma_tree_k21_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k31 = Tree.get(path="upgma_tree_k31_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k51 = Tree.get(path="upgma_tree_k51_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k61 = Tree.get(path="upgma_tree_k61_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k71 = Tree.get(path="upgma_tree_k71_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21andk51 = Tree.get(path="upgma_tree_k21andk51_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)




weighted_rf_k16 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k16, newick_busco)
unweighted_rf_k16 = treecompare.symmetric_difference(newick_sourmash_k16, newick_busco)
corr_k16, pval_k16, log_pval_k16, n_pairs = tree_distance_correlation(newick_sourmash_k16, newick_busco, method='spearman')
print(f"Weighted RF Distance (k16): {weighted_rf_k16}")
print(f"Unweighted RF Distance (k16): {unweighted_rf_k16}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k16}")
#print(f"Spearman-Rank LogP: {log_pval_k16}")

print("___________________________________")


weighted_rf_k21 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21, newick_busco)
unweighted_rf_k21 = treecompare.symmetric_difference(newick_sourmash_k21, newick_busco)
corr_k21, pval_k21, log_pval_k21, n_pairs = tree_distance_correlation(newick_sourmash_k21, newick_busco, method='spearman')
print(f"Weighted RF Distance (k21): {weighted_rf_k21}")
print(f"Unweighted RF Distance (k21): {unweighted_rf_k21}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21}")
#print(f"Spearman-Rank LogP: {log_pval_k21}")

print("___________________________________")


weighted_rf_k31 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k31, newick_busco)
unweighted_rf_k31 = treecompare.symmetric_difference(newick_sourmash_k31, newick_busco)
corr_k31, pval_k31, log_pval_k31, n_pairs = tree_distance_correlation(newick_sourmash_k31, newick_busco, method='spearman')
print(f"Weighted RF Distance (k31): {weighted_rf_k31}")
print(f"Unweighted RF Distance (k31): {unweighted_rf_k31}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k31}")
#print(f"Spearman-Rank LogP: {log_pval_k31}")

print("___________________________________")


weighted_rf_k51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k51, newick_busco)
unweighted_rf_k51 = treecompare.symmetric_difference(newick_sourmash_k51, newick_busco)
corr_k51, pval_k51, log_pval_k51, n_pairs = tree_distance_correlation(newick_sourmash_k51, newick_busco, method='spearman')
print(f"Weighted RF Distance (k51): {weighted_rf_k51}")
print(f"Unweighted RF Distance (k51): {unweighted_rf_k51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k51}")
#print(f"Spearman-Rank LogP: {log_pval_k51}")

print("___________________________________")

weighted_rf_k61 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k61, newick_busco)
unweighted_rf_k61 = treecompare.symmetric_difference(newick_sourmash_k61, newick_busco)
corr_k61, pval_k61, log_pval_k61, n_pairs = tree_distance_correlation(newick_sourmash_k61, newick_busco, method='spearman')
print(f"Weighted RF Distance (k61): {weighted_rf_k61}")
print(f"Unweighted RF Distance (k61): {unweighted_rf_k61}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k61}")
#print(f"Spearman-Rank LogP: {log_pval_k61}")

print("___________________________________")

weighted_rf_k71 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k71, newick_busco)
unweighted_rf_k71 = treecompare.symmetric_difference(newick_sourmash_k71, newick_busco)
corr_k71, pval_k71, log_pval_k71, n_pairs = tree_distance_correlation(newick_sourmash_k71, newick_busco, method='spearman')
print(f"Weighted RF Distance (k71): {weighted_rf_k71}")
print(f"Unweighted RF Distance (k71): {unweighted_rf_k71}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k71}")
#print(f"Spearman-Rank LogP: {log_pval_k71}")

print("___________________________________")


weighted_rf_k21andk51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21andk51, newick_busco)
unweighted_rf_k21andk51 = treecompare.symmetric_difference(newick_sourmash_k21andk51, newick_busco)
corr_k21andk51, pval_k21andk51, log_pval_k21andk51, n_pairs = tree_distance_correlation(newick_sourmash_k21andk51, newick_busco, method='spearman')
print(f"Weighted RF Distance (k21andk51): {weighted_rf_k21andk51}")
print(f"Unweighted RF Distance (k21andk51): {unweighted_rf_k21andk51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21andk51}")
#print(f"Spearman-Rank LogP: {log_pval_k21andk51}")


In [ ]:
results = [
    ("k16", round(weighted_rf_k16, 2), unweighted_rf_k16, round(corr_k16, 2)),
    ("k21", round(weighted_rf_k21, 2), unweighted_rf_k21, round(corr_k21, 2)),
    ("k31", round(weighted_rf_k31, 2), unweighted_rf_k31, round(corr_k31, 2)),
    ("k51", round(weighted_rf_k51, 2), unweighted_rf_k51, round(corr_k51, 2)),
    ("k61", round(weighted_rf_k61, 2), unweighted_rf_k61, round(corr_k61, 2)),
    ("k71", round(weighted_rf_k71, 2), unweighted_rf_k71, round(corr_k71, 2)),
    ("k21andk51", round(weighted_rf_k21andk51, 2), unweighted_rf_k21andk51, round(corr_k21andk51, 2))
]

# Print in Excel-ready format
print("Method\tWeighted_RF\tUnweighted_RF\tSpearman-Rank_Corr._Coef.")  # Header
for row in results:
    print(f"{row[0]}\t{row[1]}\t{row[2]}\t{row[3]}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results, columns=['Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr'])

# Add metadata columns (same value for all rows)
df['Species'] = 'Cyptococcus Complexes'
df['ANI_type'] = 'Jaccard'

# Reorder columns for better organization (optional)
df = df[['Species', 'ANI_type', 'Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr']]

# Print Excel-ready format
print("Species\tANI_type\tMethod\tWeighted_RF\tUnweighted_RF\tSpearman_Corr")
for _, row in df.iterrows():
    print(f"{row['Species']}\t{row['ANI_type']}\t{row['Method']}\t{row['Weighted_RF']}\t{row['Unweighted_RF']}\t{row['Spearman_Corr']}")

# Save as CSV for later use
df.to_csv('cryptococcus_reduced_jaccard_results.csv', index=False, sep='\t')

In [ ]:
def normalize_by_tree_length(tree):
    total = sum(edge.length for edge in tree.postorder_edge_iter() if edge.length)
    for edge in tree.postorder_edge_iter():
        if edge.length is not None:
            edge.length /= total

In [ ]:
os.chdir(SourmashTreeDir)

# Create a shared TaxonNamespace
shared_taxon_namespace = TaxonNamespace()

# Load the trees with the shared TaxonNamespace
os.chdir(SourmashTreeDir)
newick_busco = Tree.get(path="CryptococcusBUSCOs_corrected_names.tree", schema="newick", taxon_namespace=shared_taxon_namespace)

        
newick_sourmash_k16 = Tree.get(path="upgma_tree_k16_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21 = Tree.get(path="upgma_tree_k21_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k31 = Tree.get(path="upgma_tree_k31_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k51 = Tree.get(path="upgma_tree_k51_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k61 = Tree.get(path="upgma_tree_k61_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k71 = Tree.get(path="upgma_tree_k71_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21andk51 = Tree.get(path="upgma_tree_k21andk51_jaccard.newick", schema="newick", taxon_namespace=shared_taxon_namespace)



# Apply normalization to all trees
for tree in [newick_busco, newick_sourmash_k16, newick_sourmash_k21, 
             newick_sourmash_k31, newick_sourmash_k51, newick_sourmash_k61, newick_sourmash_k71, newick_sourmash_k21andk51]:
    normalize_by_tree_length(tree)

    
weighted_rf_k16 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k16, newick_busco)
unweighted_rf_k16 = treecompare.symmetric_difference(newick_sourmash_k16, newick_busco)
corr_k16, pval_k16, log_pval_k16, n_pairs = tree_distance_correlation(newick_sourmash_k16, newick_busco, method='spearman')
print(f"Weighted RF Distance (k16): {weighted_rf_k16}")
print(f"Unweighted RF Distance (k16): {unweighted_rf_k16}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k16}")
#print(f"Spearman-Rank LogP: {log_pval_k16}")

print("___________________________________")


weighted_rf_k21 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21, newick_busco)
unweighted_rf_k21 = treecompare.symmetric_difference(newick_sourmash_k21, newick_busco)
corr_k21, pval_k21, log_pval_k21, n_pairs = tree_distance_correlation(newick_sourmash_k21, newick_busco, method='spearman')
print(f"Weighted RF Distance (k21): {weighted_rf_k21}")
print(f"Unweighted RF Distance (k21): {unweighted_rf_k21}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21}")
#print(f"Spearman-Rank LogP: {log_pval_k21}")

print("___________________________________")


weighted_rf_k31 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k31, newick_busco)
unweighted_rf_k31 = treecompare.symmetric_difference(newick_sourmash_k31, newick_busco)
corr_k31, pval_k31, log_pval_k31, n_pairs = tree_distance_correlation(newick_sourmash_k31, newick_busco, method='spearman')
print(f"Weighted RF Distance (k31): {weighted_rf_k31}")
print(f"Unweighted RF Distance (k31): {unweighted_rf_k31}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k31}")
#print(f"Spearman-Rank LogP: {log_pval_k31}")

print("___________________________________")


weighted_rf_k51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k51, newick_busco)
unweighted_rf_k51 = treecompare.symmetric_difference(newick_sourmash_k51, newick_busco)
corr_k51, pval_k51, log_pval_k51, n_pairs = tree_distance_correlation(newick_sourmash_k51, newick_busco, method='spearman')
print(f"Weighted RF Distance (k51): {weighted_rf_k51}")
print(f"Unweighted RF Distance (k51): {unweighted_rf_k51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k51}")
#print(f"Spearman-Rank LogP: {log_pval_k51}")

print("___________________________________")

weighted_rf_k61 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k61, newick_busco)
unweighted_rf_k61 = treecompare.symmetric_difference(newick_sourmash_k61, newick_busco)
corr_k61, pval_k61, log_pval_k61, n_pairs = tree_distance_correlation(newick_sourmash_k61, newick_busco, method='spearman')
print(f"Weighted RF Distance (k61): {weighted_rf_k61}")
print(f"Unweighted RF Distance (k61): {unweighted_rf_k61}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k61}")
#print(f"Spearman-Rank LogP: {log_pval_k61}")

print("___________________________________")

weighted_rf_k71 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k71, newick_busco)
unweighted_rf_k71 = treecompare.symmetric_difference(newick_sourmash_k71, newick_busco)
corr_k71, pval_k71, log_pval_k71, n_pairs = tree_distance_correlation(newick_sourmash_k71, newick_busco, method='spearman')
print(f"Weighted RF Distance (k71): {weighted_rf_k71}")
print(f"Unweighted RF Distance (k71): {unweighted_rf_k71}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k71}")
#print(f"Spearman-Rank LogP: {log_pval_k71}")

print("___________________________________")


weighted_rf_k21andk51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21andk51, newick_busco)
unweighted_rf_k21andk51 = treecompare.symmetric_difference(newick_sourmash_k21andk51, newick_busco)
corr_k21andk51, pval_k21andk51, log_pval_k21andk51, n_pairs = tree_distance_correlation(newick_sourmash_k21andk51, newick_busco, method='spearman')
print(f"Weighted RF Distance (k21andk51): {weighted_rf_k21andk51}")
print(f"Unweighted RF Distance (k21andk51): {unweighted_rf_k21andk51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21andk51}")
#print(f"Spearman-Rank LogP: {log_pval_k21andk51}")


In [ ]:
results = [
    ("k16", round(weighted_rf_k16, 2), unweighted_rf_k16, round(corr_k16, 2)),
    ("k21", round(weighted_rf_k21, 2), unweighted_rf_k21, round(corr_k21, 2)),
    ("k31", round(weighted_rf_k31, 2), unweighted_rf_k31, round(corr_k31, 2)),
    ("k51", round(weighted_rf_k51, 2), unweighted_rf_k51, round(corr_k51, 2)),
    ("k61", round(weighted_rf_k61, 2), unweighted_rf_k61, round(corr_k61, 2)),
    ("k71", round(weighted_rf_k71, 2), unweighted_rf_k71, round(corr_k71, 2)),
    ("k21andk51", round(weighted_rf_k21andk51, 2), unweighted_rf_k21andk51, round(corr_k21andk51, 2))
]

# Print in Excel-ready format
print("Method\tWeighted_RF\tUnweighted_RF\tSpearman-Rank_Corr._Coef.")  # Header
for row in results:
    print(f"{row[0]}\t{row[1]}\t{row[2]}\t{row[3]}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results, columns=['Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr'])

# Add metadata columns (same value for all rows)
df['Species'] = 'Cyptococcus Complexes'
df['ANI_type'] = 'Jaccard'

# Reorder columns for better organization (optional)
df = df[['Species', 'ANI_type', 'Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr']]

# Print Excel-ready format
print("Species\tANI_type\tMethod\tWeighted_RF\tUnweighted_RF\tSpearman_Corr")
for _, row in df.iterrows():
    print(f"{row['Species']}\t{row['ANI_type']}\t{row['Method']}\t{row['Weighted_RF']}\t{row['Unweighted_RF']}\t{row['Spearman_Corr']}")

# Save as CSV for later use
df.to_csv('cryptococcus_reduced_jaccard_results.csv', index=False, sep='\t')

## ______________________________________________________________________________

## ______________________________________________________________________________

### Load the sourmash matrices and convert to trees, also prepare a combine k21 and k51 matrix (where under 95% ANI k=21, and over 95% k=51)
### This time using max containment

##### First load matrices for sourmash ANI computed by max containment

In [ ]:
# Load the k=21 and k=51 ANI matrices
k16_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k16/comprehensive_maxcontainment.npy"
k21_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k21/comprehensive_maxcontainment.npy"
k31_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k31/comprehensive_maxcontainment.npy"
k51_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k51/comprehensive_maxcontainment.npy"
k61_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k61/comprehensive_maxcontainment.npy"
k71_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k71/comprehensive_maxcontainment.npy"

ani_16_j = np.load(k16_path_j)
ani_21_j = np.load(k21_path_j)
ani_31_j = np.load(k31_path_j)
ani_51_j = np.load(k51_path_j)
ani_61_j = np.load(k61_path_j)
ani_71_j = np.load(k71_path_j)

# Load the labels for both k values
k16_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k16/comprehensive_maxcontainment.npy.labels.txt"
k21_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k21/comprehensive_maxcontainment.npy.labels.txt"
k31_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k31/comprehensive_maxcontainment.npy.labels.txt"
k51_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k51/comprehensive_maxcontainment.npy.labels.txt"
k61_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k61/comprehensive_maxcontainment.npy.labels.txt"
k71_labels_path_j = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k71/comprehensive_maxcontainment.npy.labels.txt"

def parse_labels(label_file):
    with open(label_file) as f:
        labels = [line.strip() for line in f]
    # Parse labels to get just the accession (GCA_XXXXXX.X)
    parsed_labels = []
    for label in labels:
        # Get the filename part after the last '/'
        filename = label.split('/')[-1]
        # Split by '_' and take first two parts
        accession = '_'.join(filename.split('_')[:2])
        parsed_labels.append(accession)
    return parsed_labels

k16_labels_j = parse_labels(k16_labels_path_j)
k21_labels_j = parse_labels(k21_labels_path_j)
k31_labels_j = parse_labels(k31_labels_path_j)
k51_labels_j = parse_labels(k51_labels_path_j)
k61_labels_j = parse_labels(k61_labels_path_j)
k71_labels_j = parse_labels(k71_labels_path_j)


In [ ]:
## Compute the merged ANI matrix for k=21 until ANI becomes 95%, then switch to k=51

# Verify the labels match between k21 and k51
if k21_labels_j != k51_labels_j:
    print("Warning: Label order differs between k21 and k51 matrices!")
    
    # Create a mapping from accession to index in k51 matrix
    k51_label_to_index_j = {label: i for i, label in enumerate(k51_labels_j)}
    
    # Reorder the k51 matrix to match k21's order
    reorder_indices_j = [k51_label_to_index_j[label] for label in k21_labels_j]
    ani_51_j_reorder = ani_51_j[reorder_indices_j][:, reorder_indices_j]
    
    print("Reordered k51 matrix to match k21 label order")
else:
    print("Label orders match between k21 and k51 matrices")

# Merge the matrices
k21k51_ani_j = np.where(ani_21_j > 0.95, ani_51_j_reorder, ani_21_j)

# Save the merged matrix as both .npy and .csv
output_dir = "/path/to/your/analysis_directory/FirstManuscriptTreeComparisons/SpeciesComplex_SourmashTrees/CryptococcusComplexes_Reduced/SourmashSignatures/k21k51"
os.makedirs(output_dir, exist_ok=True)
np.save(f"{output_dir}/k21and51_ANI_maxcontainment.npy", k21k51_ani_j)

# Save as CSV with parsed labels
df = pd.DataFrame(k21k51_ani_j, index=k21_labels_j, columns=k21_labels_j)
df.to_csv(f"{output_dir}/k21and51_ANI_maxcontainment_parsed.csv")

##### Now build each sourmash tree

In [ ]:
## For k=16

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_16_j

# Condense to lower-triangular format
num_taxa = len(ani_16_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(k16_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_16 = constructor.upgma(dist_matrix)
upgma_tree_16.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_16.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k16_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_16, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=21

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_21_j

# Condense to lower-triangular format
num_taxa = len(ani_21_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k21_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_21 = constructor.upgma(dist_matrix)
upgma_tree_21.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_21.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]



## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k21_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_21, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=31

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_31_j

# Condense to lower-triangular format
num_taxa = len(ani_31_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k31_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_31 = constructor.upgma(dist_matrix)
upgma_tree_31.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_31.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]



## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k31_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_31, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=51

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_51_j

# Condense to lower-triangular format
num_taxa = len(ani_51_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k51_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_51 = constructor.upgma(dist_matrix)
upgma_tree_51.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_51.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k51_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_51, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=61

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_61_j

# Condense to lower-triangular format
num_taxa = len(ani_61_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k61_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_61 = constructor.upgma(dist_matrix)
upgma_tree_61.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_61.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k61_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_61, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=71

# Convert similarity matrix to distance matrix
DistMatrix = 1 - ani_71_j

# Condense to lower-triangular format
num_taxa = len(ani_71_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k71_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_71 = constructor.upgma(dist_matrix)
upgma_tree_71.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_71.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]


## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k71_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_71, output_path, "newick")

print(f"Tree saved to: {output_path}")

In [ ]:
## For k=21 until ANI becomes 95%, then switch to k=51

# Convert similarity matrix to distance matrix
DistMatrix = 1 - k21k51_ani_j

# Condense to lower-triangular format
num_taxa = len(k21k51_ani_j)
condensed_matrix = []
for i in range(num_taxa):
    condensed_matrix.append(DistMatrix[i, :i+1].tolist())

# Create BioPython DistanceMatrix object
dist_matrix = DistanceMatrix(names=k21_labels_j, matrix=condensed_matrix)

# Build UPGMA tree
constructor = DistanceTreeConstructor()
upgma_tree_21and51 = constructor.upgma(dist_matrix)
upgma_tree_21and51.ladderize()  

# Update the leaf labels in the tree
for leaf in upgma_tree_21and51.get_terminals():
    if leaf.name in CryptococcusDictionary:
        leaf.name = CryptococcusDictionary[leaf.name]



## Save the tree 
# Define the output file path
output_path = os.path.join(SourmashTreeDir, "upgma_tree_k21andk51_maxcontainment.newick")

# Save the tree in Newick format
Phylo.write(upgma_tree_21and51, output_path, "newick")

print(f"Tree saved to: {output_path}")

## ____________________________________________________________________________________________

### Plot sourmash trees next to BUSCO trees, and compute Robinson-Foulds distances

In [ ]:
os.chdir(SourmashTreeDir)

# Create a shared TaxonNamespace
shared_taxon_namespace = TaxonNamespace()

# Load the trees with the shared TaxonNamespace
os.chdir(SourmashTreeDir)
newick_busco = Tree.get(path="CryptococcusBUSCOs_corrected_names.tree", schema="newick", taxon_namespace=shared_taxon_namespace)

        
newick_sourmash_k16 = Tree.get(path="upgma_tree_k16_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21 = Tree.get(path="upgma_tree_k21_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k31 = Tree.get(path="upgma_tree_k31_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k51 = Tree.get(path="upgma_tree_k51_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k61 = Tree.get(path="upgma_tree_k61_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k71 = Tree.get(path="upgma_tree_k71_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21andk51 = Tree.get(path="upgma_tree_k21andk51_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)




weighted_rf_k16 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k16, newick_busco)
unweighted_rf_k16 = treecompare.symmetric_difference(newick_sourmash_k16, newick_busco)
corr_k16, pval_k16, log_pval_k16, n_pairs = tree_distance_correlation(newick_sourmash_k16, newick_busco)
print(f"Weighted RF Distance (k16): {weighted_rf_k16}")
print(f"Unweighted RF Distance (k16): {unweighted_rf_k16}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k16}")
#print(f"Spearman-Rank LogP: {log_pval_k16}")

print("___________________________________")


weighted_rf_k21 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21, newick_busco)
unweighted_rf_k21 = treecompare.symmetric_difference(newick_sourmash_k21, newick_busco)
corr_k21, pval_k21, log_pval_k21, n_pairs = tree_distance_correlation(newick_sourmash_k21, newick_busco)
print(f"Weighted RF Distance (k21): {weighted_rf_k21}")
print(f"Unweighted RF Distance (k21): {unweighted_rf_k21}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21}")
#print(f"Spearman-Rank LogP: {log_pval_k21}")

print("___________________________________")


weighted_rf_k31 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k31, newick_busco)
unweighted_rf_k31 = treecompare.symmetric_difference(newick_sourmash_k31, newick_busco)
corr_k31, pval_k31, log_pval_k31, n_pairs = tree_distance_correlation(newick_sourmash_k31, newick_busco)
print(f"Weighted RF Distance (k31): {weighted_rf_k31}")
print(f"Unweighted RF Distance (k31): {unweighted_rf_k31}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k31}")
#print(f"Spearman-Rank LogP: {log_pval_k31}")

print("___________________________________")


weighted_rf_k51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k51, newick_busco)
unweighted_rf_k51 = treecompare.symmetric_difference(newick_sourmash_k51, newick_busco)
corr_k51, pval_k51, log_pval_k51, n_pairs = tree_distance_correlation(newick_sourmash_k51, newick_busco)
print(f"Weighted RF Distance (k51): {weighted_rf_k51}")
print(f"Unweighted RF Distance (k51): {unweighted_rf_k51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k51}")
#print(f"Spearman-Rank LogP: {log_pval_k51}")

print("___________________________________")

weighted_rf_k61 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k61, newick_busco)
unweighted_rf_k61 = treecompare.symmetric_difference(newick_sourmash_k61, newick_busco)
corr_k61, pval_k61, log_pval_k61, n_pairs = tree_distance_correlation(newick_sourmash_k61, newick_busco)
print(f"Weighted RF Distance (k61): {weighted_rf_k61}")
print(f"Unweighted RF Distance (k61): {unweighted_rf_k61}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k61}")
#print(f"Spearman-Rank LogP: {log_pval_k61}")

print("___________________________________")

weighted_rf_k71 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k71, newick_busco)
unweighted_rf_k71 = treecompare.symmetric_difference(newick_sourmash_k71, newick_busco)
corr_k71, pval_k71, log_pval_k71, n_pairs = tree_distance_correlation(newick_sourmash_k71, newick_busco)
print(f"Weighted RF Distance (k71): {weighted_rf_k71}")
print(f"Unweighted RF Distance (k71): {unweighted_rf_k71}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k71}")
#print(f"Spearman-Rank LogP: {log_pval_k71}")

print("___________________________________")


weighted_rf_k21andk51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21andk51, newick_busco)
unweighted_rf_k21andk51 = treecompare.symmetric_difference(newick_sourmash_k21andk51, newick_busco)
corr_k21andk51, pval_k21andk51, log_pval_k21andk51, n_pairs = tree_distance_correlation(newick_sourmash_k21andk51, newick_busco)
print(f"Weighted RF Distance (k21andk51): {weighted_rf_k21andk51}")
print(f"Unweighted RF Distance (k21andk51): {unweighted_rf_k21andk51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21andk51}")
#print(f"Spearman-Rank LogP: {log_pval_k21andk51}")


In [ ]:
results = [
    ("k16", round(weighted_rf_k16, 2), unweighted_rf_k16, round(corr_k16, 2)),
    ("k21", round(weighted_rf_k21, 2), unweighted_rf_k21, round(corr_k21, 2)),
    ("k31", round(weighted_rf_k31, 2), unweighted_rf_k31, round(corr_k31, 2)),
    ("k51", round(weighted_rf_k51, 2), unweighted_rf_k51, round(corr_k51, 2)),
    ("k61", round(weighted_rf_k61, 2), unweighted_rf_k61, round(corr_k61, 2)),
    ("k71", round(weighted_rf_k71, 2), unweighted_rf_k71, round(corr_k71, 2)),
    ("k21andk51", round(weighted_rf_k21andk51, 2), unweighted_rf_k21andk51, round(corr_k21andk51, 2))
]

# Print in Excel-ready format
print("Method\tWeighted_RF\tUnweighted_RF\tSpearman-Rank_Corr._Coef.")  # Header
for row in results:
    print(f"{row[0]}\t{row[1]}\t{row[2]}\t{row[3]}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results, columns=['Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr'])

# Add metadata columns (same value for all rows)
df['Species'] = 'Cyptococcus Complexes'
df['ANI_type'] = 'Max Containment'

# Reorder columns for better organization (optional)
df = df[['Species', 'ANI_type', 'Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr']]

# Print Excel-ready format
print("Species\tANI_type\tMethod\tWeighted_RF\tUnweighted_RF\tSpearman_Corr")
for _, row in df.iterrows():
    print(f"{row['Species']}\t{row['ANI_type']}\t{row['Method']}\t{row['Weighted_RF']}\t{row['Unweighted_RF']}\t{row['Spearman_Corr']}")

# Save as CSV for later use
df.to_csv('cryptococcus_reduced_maxcontainment_results.csv', index=False, sep='\t')

In [ ]:
os.getcwd()

In [ ]:
def normalize_by_tree_length(tree):
    total = sum(edge.length for edge in tree.postorder_edge_iter() if edge.length)
    for edge in tree.postorder_edge_iter():
        if edge.length is not None:
            edge.length /= total

In [ ]:
os.chdir(SourmashTreeDir)

# Create a shared TaxonNamespace
shared_taxon_namespace = TaxonNamespace()

# Load the trees with the shared TaxonNamespace
os.chdir(SourmashTreeDir)
newick_busco = Tree.get(path="CryptococcusBUSCOs_corrected_names.tree", schema="newick", taxon_namespace=shared_taxon_namespace)

        
newick_sourmash_k16 = Tree.get(path="upgma_tree_k16_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21 = Tree.get(path="upgma_tree_k21_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k31 = Tree.get(path="upgma_tree_k31_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k51 = Tree.get(path="upgma_tree_k51_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k61 = Tree.get(path="upgma_tree_k61_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k71 = Tree.get(path="upgma_tree_k71_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)
newick_sourmash_k21andk51 = Tree.get(path="upgma_tree_k21andk51_maxcontainment.newick", schema="newick", taxon_namespace=shared_taxon_namespace)


# Apply normalization to all trees
for tree in [newick_busco, newick_sourmash_k16, newick_sourmash_k21, 
             newick_sourmash_k31, newick_sourmash_k51, newick_sourmash_k61, newick_sourmash_k71, newick_sourmash_k21andk51]:
    normalize_by_tree_length(tree)
    

weighted_rf_k16 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k16, newick_busco)
unweighted_rf_k16 = treecompare.symmetric_difference(newick_sourmash_k16, newick_busco)
corr_k16, pval_k16, log_pval_k16, n_pairs = tree_distance_correlation(newick_sourmash_k16, newick_busco)
print(f"Weighted RF Distance (k16): {weighted_rf_k16}")
print(f"Unweighted RF Distance (k16): {unweighted_rf_k16}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k16}")
#print(f"Spearman-Rank LogP: {log_pval_k16}")

print("___________________________________")


weighted_rf_k21 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21, newick_busco)
unweighted_rf_k21 = treecompare.symmetric_difference(newick_sourmash_k21, newick_busco)
corr_k21, pval_k21, log_pval_k21, n_pairs = tree_distance_correlation(newick_sourmash_k21, newick_busco)
print(f"Weighted RF Distance (k21): {weighted_rf_k21}")
print(f"Unweighted RF Distance (k21): {unweighted_rf_k21}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21}")
#print(f"Spearman-Rank LogP: {log_pval_k21}")

print("___________________________________")


weighted_rf_k31 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k31, newick_busco)
unweighted_rf_k31 = treecompare.symmetric_difference(newick_sourmash_k31, newick_busco)
corr_k31, pval_k31, log_pval_k31, n_pairs = tree_distance_correlation(newick_sourmash_k31, newick_busco)
print(f"Weighted RF Distance (k31): {weighted_rf_k31}")
print(f"Unweighted RF Distance (k31): {unweighted_rf_k31}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k31}")
#print(f"Spearman-Rank LogP: {log_pval_k31}")

print("___________________________________")


weighted_rf_k51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k51, newick_busco)
unweighted_rf_k51 = treecompare.symmetric_difference(newick_sourmash_k51, newick_busco)
corr_k51, pval_k51, log_pval_k51, n_pairs = tree_distance_correlation(newick_sourmash_k51, newick_busco)
print(f"Weighted RF Distance (k51): {weighted_rf_k51}")
print(f"Unweighted RF Distance (k51): {unweighted_rf_k51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k51}")
#print(f"Spearman-Rank LogP: {log_pval_k51}")

print("___________________________________")

weighted_rf_k61 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k61, newick_busco)
unweighted_rf_k61 = treecompare.symmetric_difference(newick_sourmash_k61, newick_busco)
corr_k61, pval_k61, log_pval_k61, n_pairs = tree_distance_correlation(newick_sourmash_k61, newick_busco)
print(f"Weighted RF Distance (k61): {weighted_rf_k61}")
print(f"Unweighted RF Distance (k61): {unweighted_rf_k61}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k61}")
#print(f"Spearman-Rank LogP: {log_pval_k61}")

print("___________________________________")

weighted_rf_k71 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k71, newick_busco)
unweighted_rf_k71 = treecompare.symmetric_difference(newick_sourmash_k71, newick_busco)
corr_k71, pval_k71, log_pval_k71, n_pairs = tree_distance_correlation(newick_sourmash_k71, newick_busco)
print(f"Weighted RF Distance (k71): {weighted_rf_k71}")
print(f"Unweighted RF Distance (k71): {unweighted_rf_k71}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k71}")
#print(f"Spearman-Rank LogP: {log_pval_k71}")

print("___________________________________")


weighted_rf_k21andk51 = treecompare.weighted_robinson_foulds_distance(newick_sourmash_k21andk51, newick_busco)
unweighted_rf_k21andk51 = treecompare.symmetric_difference(newick_sourmash_k21andk51, newick_busco)
corr_k21andk51, pval_k21andk51, log_pval_k21andk51, n_pairs = tree_distance_correlation(newick_sourmash_k21andk51, newick_busco)
print(f"Weighted RF Distance (k21andk51): {weighted_rf_k21andk51}")
print(f"Unweighted RF Distance (k21andk51): {unweighted_rf_k21andk51}")
print(f"Spearman-Rank Correlation Coeff.: {corr_k21andk51}")
#print(f"Spearman-Rank LogP: {log_pval_k21andk51}")


In [ ]:
results = [
    ("k16", round(weighted_rf_k16, 2), unweighted_rf_k16, round(corr_k16, 2)),
    ("k21", round(weighted_rf_k21, 2), unweighted_rf_k21, round(corr_k21, 2)),
    ("k31", round(weighted_rf_k31, 2), unweighted_rf_k31, round(corr_k31, 2)),
    ("k51", round(weighted_rf_k51, 2), unweighted_rf_k51, round(corr_k51, 2)),
    ("k61", round(weighted_rf_k61, 2), unweighted_rf_k61, round(corr_k61, 2)),
    ("k71", round(weighted_rf_k71, 2), unweighted_rf_k71, round(corr_k71, 2)),
    ("k21andk51", round(weighted_rf_k21andk51, 2), unweighted_rf_k21andk51, round(corr_k21andk51, 2))
]

# Print in Excel-ready format
print("Method\tWeighted_RF\tUnweighted_RF\tSpearman-Rank_Corr._Coef.")  # Header
for row in results:
    print(f"{row[0]}\t{row[1]}\t{row[2]}\t{row[3]}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(results, columns=['Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr'])

# Add metadata columns (same value for all rows)
df['Species'] = 'Cyptococcus Complexes'
df['ANI_type'] = 'Max Containment'

# Reorder columns for better organization (optional)
df = df[['Species', 'ANI_type', 'Method', 'Weighted_RF', 'Unweighted_RF', 'Spearman_Corr']]

# Print Excel-ready format
print("Species\tANI_type\tMethod\tWeighted_RF\tUnweighted_RF\tSpearman_Corr")
for _, row in df.iterrows():
    print(f"{row['Species']}\t{row['ANI_type']}\t{row['Method']}\t{row['Weighted_RF']}\t{row['Unweighted_RF']}\t{row['Spearman_Corr']}")

# Save as CSV for later use
df.to_csv('cryptococcus_reduced_maxcontainment_results.csv', index=False, sep='\t')